# Blackberry 3DGS on Google Colab (`engines_cuda`)

Uses **PyTorch + gsplat** (CUDA). MCMC recipe toward Postshot-like quality:
`--strategy mcmc --cap-max 200000` (see `STRATEGY_PLAN.md`).

**Runtime -> GPU** required. Clone branch: `feature/colab-train`.

Mesh from splats: `engines_cuda/export/mesh_from_ckpt.py` (Open3D) - section 6.


## 1. GPU check


In [1]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print(torch.__version__, torch.cuda.get_device_name(0), "VRAM_GiB",
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


Wed Oct 28 07:51:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone repo + install deps

Branch **`feature/colab-train`** includes MCMC / antialiased / resume.


In [2]:
import os
REPO = "/content/splats"
BRANCH = "feature/colab-train"
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} --single-branch https://github.com/aidazsolt-bot/splats.git {REPO}
else:
    %cd {REPO}
    !git fetch origin {BRANCH} && git checkout {BRANCH} && git pull
%cd {REPO}
!pip -q install -U pip
!pip -q install -r engines_cuda/requirements.txt
!python -c "from gsplat.cuda._backend import _C; print('gsplat CUDA ok', _C is not None)"


Streaming output truncated to the last 5000 lines.
(    ● ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(  ●   ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
( ●    ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(●     ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
( ●    ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(  ●   ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(   ●  ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(    ● ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(     ●) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(    ● ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(   ●  ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(  ●   ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a fe

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/blackberry-splats/dataset
!wget -O /content/blackberry.zip \
    "https://github.com/danybittel/splats/releases/download/berries/blackberry.zip"
!unzip -q /content/blackberry.zip -d /content/drive/MyDrive/blackberry-splats/dataset
!ls /content/drive/MyDrive/blackberry-splats/dataset | head

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--2026-10-28 08:19:04--  https://github.com/danybittel/splats/releases/download/berries/blackberry.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1062630760/55f923f4-0af8-4db6-8076-3903dff51d34?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-28T09%3A05%3A12Z&rscd=attachment%3B+filename%3Dblackberry.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-28T08%3A04%3A28Z&ske=2026-10-28T09%3A05%3A12Z&sks=b&skv=2018-11-09&sig=pdIfKum9fZCvFz3fmLmH4ZDhrfGSN4z1mVll1E9xj7w%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXN

## 3. Mount Drive and set dataset path

Expected layout:
```
MyDrive/blackberry-splats/dataset/{cameras,images,points3D}.txt + images/
```


In [3]:
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/blackberry-splats/dataset"
# Separate out dir so resume settings stay MCMC-specific
OUT = "/content/drive/MyDrive/blackberry-splats/cuda_out_mcmc"

import os
assert os.path.isfile(f"{DATA}/cameras.txt"), f"Missing COLMAP at {DATA}"
assert os.path.isdir(f"{DATA}/images"), f"Missing images/ at {DATA}"
os.makedirs(OUT, exist_ok=True)
print("DATA", DATA)
print("OUT", OUT)


ValueError: mount failed

## 4. Smoke train (optional)

Short stack check. Uses `default` strategy + high `data-factor` to finish quickly.


In [ ]:
%cd /content/splats
!python engines_cuda/run_train.py --smoke --data "{DATA}" --out "{OUT}/smoke" \
  --data-factor 4 --strategy default


## 5. Full train (MCMC / Postshot-like candidate)

Recipe from local note / `engines_cuda/STRATEGY_PLAN.md`:
`--strategy mcmc --cap-max 200000 --max-steps 45000 --antialiased`.

| GPU | `DATA_FACTOR` | Notes |
|-----|---------------|-------|
| T4 (free) | `2` then try `1` | 200k + factor 1 may OOM |
| L4 / A100 | `1` | Closest to local note |

After disconnect: set `RESUME = True` (same flags + same `OUT`).


In [ ]:
%cd /content/splats
RESUME = False  # True after Colab disconnect / timeout
DATA_FACTOR = 1  # T4: use 2 if CUDA OOM
CAP_MAX = 200000
MAX_STEPS = 45000

cmd = (
    f'python engines_cuda/run_train.py --data "{DATA}" --out "{OUT}" '
    f'--strategy mcmc --cap-max {CAP_MAX} --max-steps {MAX_STEPS} '
    f'--data-factor {DATA_FACTOR} --antialiased'
)
if RESUME:
    cmd += " --resume"
print(cmd)
!{cmd}


## 6. List outputs / mesh from checkpoint

Mesh = Open3D Poisson from Gaussian means (`mesh_from_ckpt.py`), not PyTorch3D.


In [ ]:
!ls -lh "{OUT}"
# Optional mesh bake from a numbered ckpt:
# !python engines_cuda/export/mesh_from_ckpt.py --ckpt "{OUT}/ckpt_45000.pt" --out "{OUT}/mesh_from_splat.ply"
